# Notebook 12: SIH26131 Grand Finale: End-to-End Automated Crop Health & Advisory Pipeline
**Problem Statement:** SIH26131 — Early detection and management of crop diseases and pest infestations  
**Syllabus Integration:** CSE471 Deep Learning for Computer Vision (CO1–CO6, Units I–VI)  

---

## 1. 4-Stage Hierarchical Vision & Agronomic Architecture
```mermaid
graph TD
    A[Raw Leaf Image] --> B[Stage 1: Classification - CNN / MobileNet / ResNet]
    A --> C[Stage 2: Pest & Symptom Localization - Bounding Boxes]
    A --> D[Stage 3: Affected Region Segmentation - U-Net]
    B --> E[Pathogen & Disease ID]
    C --> F[Pest Bounding Boxes & Scores]
    D --> G[Infected Leaf Area Percentage]
    E & F & G --> H[Stage 4: Grad-CAM Explainability & IPM Advisory Engine]
    H --> I[Actionable Agronomic Report & Extension Disclaimer]
```

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.app import AgriVisionPipeline

## 2. Execute End-to-End Pipeline

In [ ]:
pipeline = AgriVisionPipeline()

sample_path = REPO_ROOT / "data" / "samples" / "classification" / "lab_plantvillage" / "Tomato___Early_Blight" / "sample_000.jpg"
img_pil = Image.open(sample_path).convert("RGB")

results = pipeline.predict(img_pil)
adv = results["advisory"]

print(f"Diagnosed Crop & Condition : {adv['crop']} - {adv['diagnosis']}")
print(f"Pathogen Agent             : {adv['pathogen']}")
print(f"Affected Leaf Surface Area : {adv['affected_leaf_area']} ({adv['severity_level']})")
print(f"Action Urgency             : {adv['action_urgency']}")

## 3. Visual Dashboard (All 4 Vision Stages)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Stage 1: Input Image
axes[0].imshow(results["raw_rgb"])
axes[0].set_title(f"Stage 1: Input\n{adv['diagnosis']} ({adv['model_confidence']})")
axes[0].axis("off")

# Stage 2: Bounding Box Detection
import matplotlib.patches as patches
axes[1].imshow(results["raw_rgb"])
axes[1].set_title(f"Stage 2: Bounding Boxes\n{len(results['detections'])} Detected Objects")
axes[1].axis("off")
for d in results["detections"]:
    b = d["box"]
    rect = patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], linewidth=2, edgecolor='yellow', facecolor='none')
    axes[1].add_patch(rect)
    axes[1].text(b[0], b[1]-5, f"{d['label']} {d['score']:.2f}", color='yellow', fontsize=10, backgroundcolor='black')

# Stage 3: U-Net Lesion Segmentation Overlay
axes[2].imshow(results["seg_overlay"])
axes[2].set_title(f"Stage 3: U-Net Segmentation\n{adv['affected_leaf_area']} Surface Area")
axes[2].axis("off")

# Stage 4: Grad-CAM Explainability
axes[3].imshow(results["gradcam_overlay"])
axes[3].set_title("Stage 4: Grad-CAM Saliency\nRed = Decisive Features")
axes[3].axis("off")

plt.tight_layout()
plt.show()

## 4. Generated Agronomic Field Advisory Report

In [ ]:
print("=" * 80)
print(f"🌾 AGRONOMIC FIELD ADVISORY: {adv['crop'].upper()} - {adv['diagnosis'].upper()}")
print("=" * 80)
print(f"• Casual Pathogen   : {adv['pathogen']}")
print(f"• Weather Trigger   : {adv['favorable_conditions']}")
print(f"• Visual Symptoms   : {adv['symptom_confirmation']}")
print(f"• Scouting Schedule : {adv['scouting_schedule']}")

print("\n🌿 CULTURAL & SANITATION PRACTICES:")
for m in adv["cultural_measures"]:
    print(f"  [+] {m}")

print("\n🐞 BIOLOGICAL / BIO-PESTICIDES:")
for b in adv["biological_controls"]:
    print(f"  [+] {b}")

print("\n🧪 CHEMICAL INTERVENTION (IF THRESHOLD EXCEEDED):")
print(f"  [!] {adv['targeted_intervention']}")

print("\n⚠️ REGULATORY & EXTENSION ADVISORY DISCLAIMER:")
print(f"  {adv['regulatory_disclaimer']}")
print("=" * 80)